# Task 3 — actual Spark optimisation, stability and explanations

Run after the four real Task 2 models. Show cluster settings, live Spark UI evidence, repeated before/after experiments, four-model perturbation results, one Spark-backed LIME explanation and a careful borough proxy audit. Never paste the road-collision notebook's timings or screenshots.

## 1️⃣ SPARK SESSION + LIVE UI LINK

Capture your **actual** Jobs/Stages timeline and task metrics while this application runs. `spark.uiWebUrl` can be null if your managed cluster exposes the UI elsewhere; ask your admin for the correct application link and capture an authentic screenshot.

In [ ]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession
ROOT = Path.cwd()
if not (ROOT / "coursework").is_dir():
    ROOT = ROOT.parent  # also works when Jupyter starts inside notebooks/
if not (ROOT / "coursework").is_dir():
    raise RuntimeError("Launch Jupyter from the TR-04 project root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from coursework.settings import (load_config, make_spark, project_path,
    read_json, require_verified_allocation, spark_configuration)
cfg = load_config()  # gitignored config/config.json: your real allocation and cluster
require_verified_allocation(cfg)  # requires YOUR independently checked Aula row and terms
spark = make_spark(cfg, "Task3")  # config-driven SparkSession.builder, not guessed Colab resources
assert isinstance(spark, SparkSession)
print("Student and allocated dataset:", cfg["student"], cfg["allocation"]["pool_reference"],
      cfg["allocation"]["dataset_name"])
print("ACTUAL Spark application / resources:", spark_configuration(spark))
print("Spark UI for THIS application:", spark.sparkContext.uiWebUrl)


## 2️⃣ READ REAL MODELS + PROCESSED DATA

Verify four observed CV results and saved pipelines before rerunning expensive analysis. The Task 3 benchmarks operate on the Task 1 Parquet, not a 50k-row sklearn extract.

In [ ]:
from coursework.data import load_processed
task2 = read_json(project_path(cfg, "results_dir") / "task2.json")
assert task2["status"] == "observed" and len(task2["models"]) == 4
processed = load_processed(spark, cfg)
print("Four fitted Spark models:", [m["name"] for m in task2["models"]])
print("CV-selected model:", task2["recommended_model"])


## 3️⃣ MEASURE SHUFFLE, CACHE, SCALABILITY + FOUR-MODEL STABILITY

The code times **identical groupBy actions** under two shuffle settings, separately logs uncached/cached repeats **and cache-fill cost**, runs 10/25/50/100%-fraction grouped queries, and applies the SAME frozen +1-hour/masked-borough perturbation to Nov–Dec TEST inputs for ALL FOUR saved classifiers. Extra retraining trials on Jan–Sep are separately reported, not substituted for the official test-data perturbation. This step also computes Spark-backed LIME and borough audit.

In [ ]:
from pyspark.storagelevel import StorageLevel
from coursework.task3 import (benchmark_optimisations, benchmark_scalability,
    test_perturbation_analysis, stability_analysis, explain_with_lime,
    fairness_by_borough, run_task3)
print("Benchmark uses:", StorageLevel.MEMORY_AND_DISK)
print("Keep the Spark UI open while this assessed computation runs.")
observed_3 = run_task3(spark, cfg)
assert observed_3["status"] == "observed"
print("Measured optimisation arms:", len(observed_3["optimisations"]))
print("Measured stability models:", list(observed_3["stability"]))


## 4️⃣ ACTUAL OPTIMISATION RESULTS (NOT A CLAIMED SPEEDUP)

Compare identical actions under different shuffle partitions; cache **fill plus reuse** may be slower overall. Setting AQE skew-join alone does not prove a groupBy skew was removed. Record the hardware, spill and stage ID yourself.

In [ ]:
import pandas as pd
from IPython.display import display
bench_rows = []
for arm in observed_3["optimisations"]:
    if arm["change"] == "shuffle_partition_count":
        bench_rows.append({"experiment": "shuffle", "setting": arm["partitions"],
                           "repeat_median_s": arm["median_seconds"]})
    else:
        bench_rows.extend([
            {"experiment": "cache", "setting": "uncached repeats",
             "repeat_median_s": arm["median_before_seconds"],
             "total_s": arm["uncached_total_seconds"]},
            {"experiment": "cache", "setting": "cached repeats",
             "repeat_median_s": arm["median_after_seconds"],
             "fill_s": arm["cache_fill_seconds"],
             "total_s": arm["cache_total_seconds_including_fill"]},
        ])
display(pd.DataFrame(bench_rows))
print("Do NOT assert a benefit unless measured total and repeat costs support one.")


## 5️⃣ SCALABILITY QUERY CURVE (WITH I/O LIMITATION)

10/25/50/100% **sampled Jan–Sep grouped-query** results; even small samples scan the source Parquet, so the curve is not linear hardware scaling or model-fit scaling.

In [ ]:
import matplotlib.pyplot as plt
scaling = pd.DataFrame(observed_3["scalability_query_benchmarks"])
display(scaling[["sample_fraction", "rows", "count_seconds", "group_by_seconds"]])
ax = scaling.plot(x="rows", y="group_by_seconds", marker="o", legend=False,
                  figsize=(7, 3.5), color="#168a85")
ax.set(title="Measured Spark groupBy wall time — training-data fractions",
       xlabel="Sampled rows (input Parquet still scanned)", ylabel="GroupBy seconds")
plt.tight_layout(); plt.show()


## 6️⃣ PERTURB NOV–DEC TEST INPUTS — FOUR-MODEL ΔF1/ΔROC-AUC

The attached official answer sheet requires a **test-data perturbation**, not only retraining sensitivity. Read the exact shared Nov–Dec perturbation protocol and F1/ROC-AUC delta and rank for each model. Labels and October-selected thresholds stay fixed; Nov–Dec is still NOT used for selecting a new model.

In [ ]:
print("Frozen held-out perturbation:", observed_3["test_perturbation_protocol"])
stability_rows = []
for model_name, info in observed_3["stability"].items():
    stability_rows.append({"model": model_name,
        "baseline_F1": info["baseline_metrics"]["positive_f1"],
        "perturbed_F1": info["perturbed_metrics"]["positive_f1"],
        "delta_F1": info["signed_deltas"]["positive_f1"],
        "delta_ROC_AUC": info["signed_deltas"]["auc_roc"],
        "rank": info["rank"]})
stability_table = pd.DataFrame(stability_rows).sort_values("rank")
display(stability_table)
print("Most/least robust under THIS predeclared input-error scenario:",
      observed_3["most_stable"], observed_3["least_stable"])
if observed_3["retraining_stability_supplement"]:
    print("Supplementary Jan-Sep retraining sensitivity (DIFFERENT question):")
    display(pd.DataFrame([{"model": name,
        "mean_abs_october_PR_delta": data["mean_abs_auc_pr_delta"]}
        for name, data in observed_3["retraining_stability_supplement"].items()]))
else:
    print("Extra retraining sensitivity disabled (not required; 16 additional fits).")


## 7️⃣ LIME ON A SAVED SPARK MODEL (ONE TRIP)

One genuinely fitted Spark PipelineModel scores LIME perturbations; the displayed weights explain one case, NOT global causality. If the local surrogate has weak fidelity R², say so rather than claiming a reliable explanation.

In [ ]:
from IPython.display import Image, display
lime = observed_3["explainability"]
print("Model and method:", lime["explained_model"], lime["method"])
print("Neighbour queries / local fidelity R²:",
      lime["neighbour_queries"], lime["local_surrogate_fidelity_r2"])
display(pd.DataFrame(lime["feature_weights"]))
display(Image(filename=str(project_path(cfg, "results_dir") / "task3_lime.png")))


## 8️⃣ BOROUGH AUDIT + BIAS LIMITATION

Report observed prevalence AND prediction rate, FPR and TPR for sufficiently large borough groups. Geography is a socioeconomic proxy, **not** a protected characteristic or proof of unlawful disparate impact; card-only tips have selection bias.

In [ ]:
groups = pd.DataFrame(observed_3["fairness"]["groups"])
print("Minimum published group size:", observed_3["fairness"]["minimum_group_size"])
print("Suppressed groups:", observed_3["fairness"]["small_groups_suppressed"])
print("Named risk:", observed_3["named_bias_risk"])
display(groups)
if not groups.empty:
    ax = groups.set_index("borough")[["observed_positive_rate",
        "predicted_positive_rate"]].plot.bar(figsize=(8, 3.5), color=["#315c84", "#dc784d"])
    ax.set(ylabel="Rate", title="Observed vs predicted positive rates — borough proxy audit")
    plt.tight_layout(); plt.show()


## 9️⃣ SPARK UI SCREENSHOT + YOUR DIAGNOSIS

Save a **real** stage screenshot as `results/task3_spark_ui.png` (do not draw or generate one). In your own report document application/stage ID, slowest vs median task, shuffle read/write, spill, bottleneck, and whether your measured optimisation actually helped. The screenshot gate below warns if evidence is absent.

In [ ]:
ui_image = project_path(cfg, "results_dir") / "task3_spark_ui.png"
if ui_image.is_file():
    display(Image(filename=str(ui_image)))
else:
    print("MISSING: capture YOUR university Spark UI Stage screenshot at", ui_image)
print("Saved measured Task 3 JSON:", project_path(cfg, "results_dir") / "task3.json")
print("Write YOUR stage/skew/cost explanation. Do not infer a speedup from toggling a flag.")


## 🔟 TASK 3 RESULT + METHODOLOGICAL REFLECTION

Use your measured timings, four stability results, LIME fidelity and suppressed borough rates to write a critical account. Explain why a cached repeat may appear fast yet increase total elapsed time, why groupBy sampling isn't hardware scaling, and why borough/payments limit a fairness claim. This is **not** a generated essay.

In [ ]:
print("Task 3 observed status:", observed_3["status"])
print("Saved real Spark UI screenshot:", observed_3["spark_ui_screenshot"])
print("Testable mitigation (not already proven):", observed_3["fairness_mitigation_to_test"])
print("LIME caveat:", observed_3["explainability"]["limitation"])
print("Report your own observations with stage IDs, measured deltas and limitations.")


**Task 3 completion:** manually inspect genuine Spark UI evidence, interpret measured trade-offs and cash-tip selection limitations, commit your observed Task 3 work, and proceed to the four real Tableau dashboards.